## Array Type Columns in Spark Dataframes

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Array Type Columns in Spark Dataframes").getOrCreate()

24/02/18 20:28:18 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/02/18 20:28:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/02/18 20:28:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/02/18 20:28:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/02/18 20:28:20 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/02/18 20:28:20 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
24/02/18 20:28:20 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
24/02/18 20:28:20 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
24

In [16]:
import datetime
users_list = [
    {
        'id': 1,
        'first_name': 'John',
        'last_name': 'Doe',
        'email': 'john.doe@example.com',
        'phone_numer' : ['+971 54 7400113', '+92 300 5638080'],
        'is_customer': True,
        'amount_paid': 123.45,
        'customer_from': datetime.date(2023, 1, 15),  # Date in YYYY-MM-DD format
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)  # Timestamp in YYYY-MM-DD HH:MM:SS format
    },
    {
        'id': 2,
        'first_name': 'Jane',
        'last_name': 'Smith',
        'email': 'jane.smith@example.com',
        'phone_numer' : ['+971 99 7666666', '+92 300 0000000'],
        'is_customer': True,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    },
    {
        'id': 3,
        'first_name': 'Jack',
        'last_name': 'Alpha',
        'email': 'jack.alpha@example.com',
        'phone_numer' : ['+971 99 7666555'],
        'is_customer': False,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    }
    # Add more user dictionaries as needed
]

In [17]:
from pyspark.sql import Row

In [35]:
users_df = spark.createDataFrame([Row(**users) for users in users_list])

In [21]:
users_df.show(truncate=False)

+---+----------+---------+----------------------+----------------------------------+-----------+-----------+-------------+-------------------+
|id |first_name|last_name|email                 |phone_numer                       |is_customer|amount_paid|customer_from|last_updated_ts    |
+---+----------+---------+----------------------+----------------------------------+-----------+-----------+-------------+-------------------+
|1  |John      |Doe      |john.doe@example.com  |[+971 54 7400113, +92 300 5638080]|true       |123.45     |2023-01-15   |2024-01-21 20:05:46|
|2  |Jane      |Smith    |jane.smith@example.com|[+971 99 7666666, +92 300 0000000]|true       |0.0        |2022-11-10   |2024-01-21 20:05:46|
|3  |Jack      |Alpha    |jack.alpha@example.com|[+971 99 7666555]                 |false      |0.0        |2022-11-10   |2024-01-21 20:05:46|
+---+----------+---------+----------------------+----------------------------------+-----------+-----------+-------------+-------------------+

In [22]:
users_df.select('id', 'first_name', 'phone_numer').show(truncate=False)

+---+----------+----------------------------------+
|id |first_name|phone_numer                       |
+---+----------+----------------------------------+
|1  |John      |[+971 54 7400113, +92 300 5638080]|
|2  |Jane      |[+971 99 7666666, +92 300 0000000]|
|3  |Jack      |[+971 99 7666555]                 |
+---+----------+----------------------------------+



In [23]:
users_df.dtypes

[('id', 'bigint'),
 ('first_name', 'string'),
 ('last_name', 'string'),
 ('email', 'string'),
 ('phone_numer', 'array<string>'),
 ('is_customer', 'boolean'),
 ('amount_paid', 'double'),
 ('customer_from', 'date'),
 ('last_updated_ts', 'timestamp')]

In [24]:
users_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_numer: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- is_customer: boolean (nullable = true)
 |-- amount_paid: double (nullable = true)
 |-- customer_from: date (nullable = true)
 |-- last_updated_ts: timestamp (nullable = true)



In [26]:
users_df.columns

['id',
 'first_name',
 'last_name',
 'email',
 'phone_numer',
 'is_customer',
 'amount_paid',
 'customer_from',
 'last_updated_ts']

In [27]:
from pyspark.sql.functions import explode

In [39]:
# if you use users_df = users_df.withColumn() ..., it convertes the users_df from df to NoneType

users_df.withColumn('phone_number', explode('phone_numer')) \
    .drop('phone_numer') \
    .show()

+---+----------+---------+--------------------+-----------+-----------+-------------+-------------------+---------------+
| id|first_name|last_name|               email|is_customer|amount_paid|customer_from|    last_updated_ts|   phone_number|
+---+----------+---------+--------------------+-----------+-----------+-------------+-------------------+---------------+
|  1|      John|      Doe|john.doe@example.com|       true|     123.45|   2023-01-15|2024-01-21 20:05:46|+971 54 7400113|
|  1|      John|      Doe|john.doe@example.com|       true|     123.45|   2023-01-15|2024-01-21 20:05:46|+92 300 5638080|
|  2|      Jane|    Smith|jane.smith@exampl...|       true|        0.0|   2022-11-10|2024-01-21 20:05:46|+971 99 7666666|
|  2|      Jane|    Smith|jane.smith@exampl...|       true|        0.0|   2022-11-10|2024-01-21 20:05:46|+92 300 0000000|
|  3|      Jack|    Alpha|jack.alpha@exampl...|      false|        0.0|   2022-11-10|2024-01-21 20:05:46|+971 99 7666555|
+---+----------+--------

In [37]:
type(users_df)

pyspark.sql.dataframe.DataFrame

In [38]:
users_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_numer: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- is_customer: boolean (nullable = true)
 |-- amount_paid: double (nullable = true)
 |-- customer_from: date (nullable = true)
 |-- last_updated_ts: timestamp (nullable = true)



In [41]:
from pyspark.sql.functions import col

In [45]:
users_df. \
    select('id', col('phone_numer')[0].alias('UAE Mobile'), col('phone_numer')[1].alias('PK Mobile')). \
    show()

+---+---------------+---------------+
| id|     UAE Mobile|      PK Mobile|
+---+---------------+---------------+
|  1|+971 54 7400113|+92 300 5638080|
|  2|+971 99 7666666|+92 300 0000000|
|  3|+971 99 7666555|           NULL|
+---+---------------+---------------+



In [46]:
from pyspark.sql.functions import explode_outer

In [48]:
users_df.withColumn('phone_numer', explode_outer('phone_numer')). \
    show()

+---+----------+---------+--------------------+---------------+-----------+-----------+-------------+-------------------+
| id|first_name|last_name|               email|    phone_numer|is_customer|amount_paid|customer_from|    last_updated_ts|
+---+----------+---------+--------------------+---------------+-----------+-----------+-------------+-------------------+
|  1|      John|      Doe|john.doe@example.com|+971 54 7400113|       true|     123.45|   2023-01-15|2024-01-21 20:05:46|
|  1|      John|      Doe|john.doe@example.com|+92 300 5638080|       true|     123.45|   2023-01-15|2024-01-21 20:05:46|
|  2|      Jane|    Smith|jane.smith@exampl...|+971 99 7666666|       true|        0.0|   2022-11-10|2024-01-21 20:05:46|
|  2|      Jane|    Smith|jane.smith@exampl...|+92 300 0000000|       true|        0.0|   2022-11-10|2024-01-21 20:05:46|
|  3|      Jack|    Alpha|jack.alpha@exampl...|+971 99 7666555|      false|        0.0|   2022-11-10|2024-01-21 20:05:46|
+---+----------+--------